# 📘 Notebook 16: The Transformer Architecture

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module E: Transformers & Fine-Tuning · Notebook 1 of 3 in this module · (16 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Understand **self-attention** in detail and implement it in NumPy
- Explain **queries, keys, and values** and the scaled dot-product formula
- Describe **multi-head attention** and why multiple heads help
- Understand why **positional encoding** is necessary
- Assemble these into the overall **Transformer** design

> **Prerequisites:** Notebook 15 (attention intuition), and the PyTorch basics from Module D.
>
> 🔭 **The architecture behind modern AI.** The Transformer, introduced in 2017, is the foundation of essentially every large language model today. The paper's title, *Attention Is All You Need*, captures the key claim: with attention done well, you do not need recurrence at all. This notebook opens the box.

> ℹ️ **Setup note.** `import piplite; await piplite.install(['numpy','torch','matplotlib'])`

## 1. Self-attention

### From attention to *self*-attention
In Notebook 15, a query looked at a separate set of inputs. In **self-attention**, every element of a sequence attends to **every other element of the same sequence**, including itself. This lets each word's representation be enriched by the context of all the other words. ‘It’ in a sentence can attend to the noun it refers to, however far away.

### Queries, Keys, and Values
Self-attention uses a database analogy. For each input element we compute three vectors by multiplying it with learned matrices:
- a **Query (Q)**, what this element is looking for;
- a **Key (K)**, what this element offers, used for matching;
- a **Value (V)**, the actual information this element contributes.

Each query is matched against all keys (by dot product) to decide how much of each value to take. The matching is exactly the attention-weight idea from Notebook 15, now done for every position against every position.

### The scaled dot-product attention formula
The whole mechanism is captured in one elegant equation:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left( \frac{Q K^{T}}{\sqrt{d_k}} \right) V$$

Reading it piece by piece:
- $QK^{T}$, every query dotted with every key: the raw match scores.
- $\div \sqrt{d_k}$, scale down by the square root of the key dimension, to keep the numbers in a stable range (the *scaled* part).
- $\text{softmax}$, turn scores into weights that sum to 1, per query.
- $\times V$, take the weighted combination of values.

Let us implement it directly:

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def self_attention(X, Wq, Wk, Wv):
    Q = X @ Wq          # queries
    K = X @ Wk          # keys
    V = X @ Wv          # values
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)     # scaled match scores
    weights = softmax(scores, axis=-1)  # attention weights (rows sum to 1)
    output = weights @ V                # weighted combination of values
    return output, weights

np.random.seed(42)
# A sequence of 3 "tokens", each a 4-dimensional vector
X = np.random.randn(3, 4)
Wq = np.random.randn(4, 4); Wk = np.random.randn(4, 4); Wv = np.random.randn(4, 4)

out, attn = self_attention(X, Wq, Wk, Wv)
print("Attention weight matrix (each row sums to 1):")
print(attn.round(3))
print("\nOutput shape:", out.shape)

Each **row** of the weight matrix shows how one token distributes its attention across all three tokens. Every token's new representation (`out`) is a blend of all the values, weighted by relevance. That is self-attention in full.

> ⚠️ **Common pitfalls**
>
> - The scaling factor $\sqrt{d_k}$ is easy to omit but important: without it, large dot products push softmax into regions with tiny gradients, hurting training.
> - Watch the matrix shapes: $Q K^T$ must be (sequence × sequence). Shape-tracing (Notebook 05's golden rule) remains essential.

## 2. Multi-head attention

### Why more than one head
A single attention computation produces one 'view' of the relationships in the sequence. **Multi-head attention** runs several attention computations (*heads*) in parallel, each with its own learned Q/K/V matrices, then concatenates their results. Different heads can specialise, one might track grammatical subject-verb links, another might track which adjective modifies which noun. Together they capture a richer picture than any single head could.

In [ ]:
def multi_head_attention(X, n_heads=2):
    d_model = X.shape[-1]
    d_head = d_model // n_heads
    rng = np.random.default_rng(0)
    head_outputs = []
    for _ in range(n_heads):
        Wq = rng.standard_normal((d_model, d_head))
        Wk = rng.standard_normal((d_model, d_head))
        Wv = rng.standard_normal((d_model, d_head))
        out, _ = self_attention(X, Wq, Wk, Wv)
        head_outputs.append(out)
    return np.concatenate(head_outputs, axis=-1)   # join the heads back together

X = np.random.randn(3, 4)
result = multi_head_attention(X, n_heads=2)
print("Multi-head output shape:", result.shape)

## 3. Positional encoding

### A subtle but crucial gap
Self-attention treats the input as a **set**, it computes the same thing regardless of word order, because it has no built-in notion of position. But order matters in language (recall *‘dog bites man’* vs *‘man bites dog’* from Notebook 15). Transformers solve this by **adding positional encodings**: a distinct pattern added to each token's representation that tells the model *where* in the sequence that token sits.

The original design uses sine and cosine waves of different frequencies, giving every position a unique, smoothly varying signature:

In [ ]:
import matplotlib.pyplot as plt

def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    angle = pos / np.power(10000, (2 * (i // 2)) / d_model)
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(angle[:, 0::2])   # even dimensions: sine
    pe[:, 1::2] = np.cos(angle[:, 1::2])   # odd dimensions: cosine
    return pe

pe = positional_encoding(seq_len=50, d_model=64)
plt.figure(figsize=(8, 4))
plt.imshow(pe, cmap="RdBu", aspect="auto")
plt.title("Positional encodings (each row is one position)")
plt.xlabel("encoding dimension"); plt.ylabel("position in sequence")
plt.colorbar(); plt.show()

Each row is a unique fingerprint for a position. Adding these to the token vectors lets attention reason about *order* as well as *content*.

## 4. Putting it together: the Transformer block

A Transformer is built by stacking identical **blocks**, each combining the pieces above with two standard tricks from earlier in the course:

```
Input + positional encoding
  -> Multi-head self-attention
  -> Add & Normalise          (a residual connection + layer norm)
  -> Feed-forward network      (ordinary dense layers, Notebook 12-13)
  -> Add & Normalise
  (repeat the block N times)
```

- The **feed-forward network** inside each block is just the fully-connected layers you built in Module D.
- **Residual connections** (the 'Add') add a layer's input to its output, helping gradients flow in deep stacks, the same idea that powers ResNet from Notebook 14.

### Two families
- **Encoder** stacks (e.g. BERT) read a whole sequence at once to *understand* it, good for classification.
- **Decoder** stacks (e.g. GPT) generate text one token at a time, good for *producing* language.

Everything is assembled from attention, dense layers, normalisation, and residual connections, all components you have now seen.

> 🧠 **You have the whole picture.** A large language model is, in essence, a very large stack of these blocks, trained by the same gradient-descent-with-backprop loop from Notebook 13, on enormous amounts of text. The scale is staggering; the *ingredients* are the ones you have been building since Notebook 05.

---
## ✏️ Exercises

### Exercise 1
In the `self_attention` function, verify numerically that **each row** of the returned attention-weight matrix sums to 1. Why must this be true?

In [ ]:
import numpy as np
# (Re-use self_attention, X, Wq, Wk, Wv from earlier.)
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
out, attn = self_attention(X, Wq, Wk, Wv)
print(attn.sum(axis=1))   # each row -> 1.0
```

*Softmax normalises each row to sum to 1, so each token's attention is a proper weighted average (a distribution) over all tokens.*
</details>

### Exercise 2
Generate positional encodings for `seq_len=10, d_model=16` and confirm that **no two positions** have identical encoding vectors (so the model can always tell positions apart). *Hint: compare rows.*

In [ ]:
import numpy as np
# (Re-use positional_encoding.)
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
pe = positional_encoding(10, 16)
# Check all pairs of rows are different:
unique = len({tuple(row.round(6)) for row in pe})
print('Distinct position encodings:', unique, 'out of', pe.shape[0])
```
</details>

## 🔑 Key takeaways

- **Self-attention** lets every token attend to every other, via **Q**, **K**, **V** vectors and a scaled dot product.
- The formula $\text{softmax}(QK^T/\sqrt{d_k})\,V$ is the entire mechanism; it is built from dot products and softmax.
- **Multi-head attention** runs several attention views in parallel to capture richer relationships.
- **Positional encoding** injects word order, which attention alone lacks.
- A **Transformer** stacks blocks of attention + feed-forward + residual/normalisation; encoders understand, decoders generate.

---
**Next:** *Notebook 17: Pretrained Models & Transfer Learning*, where we stop building from scratch and stand on the shoulders of giant models.

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*